# Committee of GNN Judges + Qwen3-4B LoRA Colab

**"Why not leverage multiple GNN-as-Judge?"**

This notebook extends the original GNN-as-Judge framework by replacing the single GNN judge
with a **heterogeneous committee** of GNNs (GCN, GAT, GraphSAGE).

### Key differences from single-judge baseline:
1. **Robust Agreement Set**: only nodes where ALL GNNs + LLM unanimously agree
2. **Uncertainty-aware Disagreement Set**: majority GNNs agree on class X, LLM disagrees,
   AND ensemble variance is low (high consensus → true hard sample)
3. **Ensemble preference score**: averaged across all judges, not a single biased model

### Pipeline:
1. Few-shot graph split + train GNN committee (GCN, GAT, SAGE)
2. Qwen3-4B 4-bit QLoRA initial SFT on few-shot labels
3. Influential node selection (graph structure)
4. LLM predictions on selected nodes
5. **Committee-of-Judges**: ensemble agreement/disagreement → WSFT + DPO data
6. WSFT fine-tuning, optional DPO preference update
7. Final evaluation (accuracy, macro-F1)

## 0. Runtime

Colab: `Runtime → Change runtime type → GPU (T4)`.

In [ ]:
#@title 1. Repo clone
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/tally0818/CS471.git"  #@param {type:"string"}
REPO_DIR = "/content/GNN-as-Judge"  #@param {type:"string"}

repo_dir = Path(REPO_DIR)
if not repo_dir.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)

candidates = [repo_dir]
candidates += [p.parent.parent for p in repo_dir.glob("**/GNN/main.py")]
project_dir = next((p for p in candidates if (p / "GNN" / "main.py").exists()), None)
if project_dir is None:
    raise RuntimeError(f"GNN-as-Judge project root not found under {repo_dir}")

os.chdir(project_dir)
print("PROJECT_DIR =", Path.cwd())

In [ ]:
#@title 2. Dependencies
import subprocess
import sys
import torch

def pip_install(args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + list(args))

pip_install([
    "-U",
    "transformers>=4.51.0",
    "accelerate>=0.34.0",
    "peft>=0.13.0",
    "bitsandbytes>=0.43.1",
    "datasets>=2.16.0",
    "gdown",
    "ogb",
    "networkx",
    "scikit-learn",
    "tqdm",
])

torch_ver = torch.__version__.split("+")[0]
cuda_tag = "cpu"
if torch.cuda.is_available() and torch.version.cuda:
    cuda_tag = "cu" + torch.version.cuda.replace(".", "")
wheel_url = f"https://data.pyg.org/whl/torch-{torch_ver}+{cuda_tag}.html"

try:
    pip_install(["torch-geometric", "pyg-lib", "torch-scatter", "torch-sparse", "-f", wheel_url])
except subprocess.CalledProcessError:
    print("PyG CUDA wheel install failed; falling back to torch-geometric only.")
    pip_install(["torch-geometric"])

print("torch:", torch.__version__, "cuda:", torch.version.cuda,
      "gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
#@title 3. Experiment config
from pathlib import Path
import torch

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"  #@param {type:"string"}
DATASET = "cora"  #@param ["cora", "citeseer", "pubmed", "arxiv"]
SHOTS = 3  #@param {type:"integer"}
SEED = 42  #@param {type:"integer"}
LORA_RANK = 8  #@param {type:"integer"}
LORA_ALPHA = 16  #@param {type:"integer"}
MAX_SELECTED_NODES = 60  #@param {type:"integer"}
MAX_TEST_SAMPLES = 100  #@param {type:"integer"}
INITIAL_SFT_EPOCHS = 3  #@param {type:"number"}
WSFT_EPOCHS = 1  #@param {type:"number"}
RUN_DPO_STAGE = True  #@param {type:"boolean"}
DPO_EPOCHS = 1  #@param {type:"integer"}
CUTOFF_LEN = 1536  #@param {type:"integer"}

# === Committee-specific config ===
GNN_TYPES = ["GCN", "GAT", "SAGE"]  #@param {type:"raw"}
GNN_HIDDEN_DIM = 64  #@param {type:"integer"}
GNN_N_LAYERS = 2  #@param {type:"integer"}
GNN_EPOCHS = 200  #@param {type:"integer"}
GNN_PATIENCE = 50  #@param {type:"integer"}
CONFIDENCE_THRESHOLD = 0.5  #@param {type:"number"}
VARIANCE_THRESHOLD = 0.1  #@param {type:"number"}
MIN_GNN_AGREEMENT = None  #@param {type:"raw"}
RETRAIN_EPOCHS = 50  #@param {type:"integer"}

BASE_DIR = Path.cwd()
DATASET_DIR = BASE_DIR / "datasets"
RUN_ID = f"{DATASET}_{SHOTS}shot_seed{SEED}_committee"
WORK_DIR = BASE_DIR / "results" / "colab_committee_judge" / RUN_ID
DATA_DIR = WORK_DIR / "data"
GNN_SAVE_DIR = WORK_DIR / "gnn_committee"
SFT_ADAPTER_DIR = WORK_DIR / "qwen3_lora8_initial_sft"
WSFT_ADAPTER_DIR = WORK_DIR / "qwen3_lora8_committee_wsft"
DPO_ADAPTER_DIR = WORK_DIR / "qwen3_lora8_committee_dpo"

for p in [DATASET_DIR, WORK_DIR, DATA_DIR, GNN_SAVE_DIR,
          SFT_ADAPTER_DIR, WSFT_ADAPTER_DIR, DPO_ADAPTER_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("RUN_ID =", RUN_ID)
print("WORK_DIR =", WORK_DIR)
print("DEVICE =", DEVICE)
print("GNN committee:", GNN_TYPES)

In [ ]:
#@title 4. Graph dataset download
import gdown
import json
import shutil
import tarfile
import zipfile
from pathlib import Path

dataset_pt = DATASET_DIR / f"{DATASET}.pt"
if not dataset_pt.exists():
    url = "https://drive.google.com/file/d/14GmRVwhP1pUD_OIhoJU3oATZWTnklhPG/view?usp=sharing"
    download_path = Path("/content/gnn_as_judge_datasets_download")
    unpack_dir = Path("/content/gnn_as_judge_datasets_unpacked")
    unpack_dir.mkdir(parents=True, exist_ok=True)
    gdown.download(url=url, output=str(download_path), quiet=False)

    if zipfile.is_zipfile(download_path):
        with zipfile.ZipFile(download_path) as zf:
            zf.extractall(unpack_dir)
    elif tarfile.is_tarfile(download_path):
        with tarfile.open(download_path) as tf:
            tf.extractall(unpack_dir)
    elif download_path.suffix == ".pt":
        shutil.copy2(download_path, DATASET_DIR / download_path.name)
    else:
        print("Downloaded file is not an archive; searching for .pt files.")

    for pt in list(unpack_dir.rglob("*.pt")) + list(Path("/content").glob("*.pt")):
        shutil.copy2(pt, DATASET_DIR / pt.name)

if not dataset_pt.exists():
    raise FileNotFoundError(f"{dataset_pt} not found.")

print("Available graph datasets:", sorted(p.name for p in DATASET_DIR.glob("*.pt")))

In [ ]:
#@title 5. Train GNN Committee (GCN + GAT + SAGE)
import sys, os
sys.path.insert(0, str(BASE_DIR))

from common import create_few_shot_dataset, set_seed
from committee_judge import train_gnn_committee

set_seed(SEED)
graph_data = create_few_shot_dataset(
    DATASET, shots=SHOTS, seed=SEED, device=DEVICE, path_prefix=str(BASE_DIR)
)

committee = train_gnn_committee(
    graph_data=graph_data,
    gnn_types=GNN_TYPES,
    hidden_dim=GNN_HIDDEN_DIM,
    n_layers=GNN_N_LAYERS,
    device=DEVICE,
    epochs=GNN_EPOCHS,
    patience=GNN_PATIENCE,
    save_dir=str(GNN_SAVE_DIR),
    dataset_name=DATASET,
    shots=SHOTS,
)

print(f"\nCommittee trained: {list(committee.keys())}")
print("Model files:", list(GNN_SAVE_DIR.glob("*.pt")))

In [ ]:
#@title 6. SFT / val / test / unlabeled JSON creation
import subprocess
import sys

sft_output_base = DATA_DIR / f"{DATASET}_sft.json"
create_sft_cmd = [
    sys.executable,
    "create_sft.py",
    "--dataset", DATASET,
    "--output", str(sft_output_base),
    "--shots", str(SHOTS),
    "--seed", str(SEED),
    "--max_test_samples", str(MAX_TEST_SAMPLES),
    "--device", DEVICE,
    "--path_prefix", str(BASE_DIR),
]
subprocess.run(create_sft_cmd, cwd=str(BASE_DIR), check=True)

SFT_PREFIX = f"{DATASET}_sft_{SHOTS}_shot"
SFT_TRAIN_JSON = DATA_DIR / f"{SFT_PREFIX}_train.json"
SFT_VAL_JSON = DATA_DIR / f"{SFT_PREFIX}_val.json"
SFT_TEST_JSON = DATA_DIR / f"{SFT_PREFIX}_test.json"
UNLABELED_JSON = DATA_DIR / f"{SFT_PREFIX}_unlabeled.json"
UNLABELED_NODE_IDS_JSON = DATA_DIR / f"{DATASET}_{SHOTS}_shot_unlabeled_node_ids.json"

for p in [SFT_TRAIN_JSON, SFT_TEST_JSON, UNLABELED_JSON, UNLABELED_NODE_IDS_JSON]:
    if not p.exists():
        raise FileNotFoundError(p)
    print(p.name, "exists")

In [ ]:
#@title 7. LLM utility functions
import gc
import json
import math
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer, TrainingArguments

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def assistant_text(row):
    for turn in row.get("conversations", []):
        if turn.get("from") == "gpt":
            return turn.get("value", "")
    return ""

def user_text(row):
    for turn in row.get("conversations", []):
        if turn.get("from") == "human":
            return turn.get("value", "")
    return ""

def tokenize_chat_example(tokenizer, prompt, answer, max_length):
    user_messages = [{"role": "user", "content": prompt}]
    full_messages = [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": answer},
    ]
    prompt_text = tokenizer.apply_chat_template(user_messages, tokenize=False, add_generation_prompt=True)
    full_text = tokenizer.apply_chat_template(full_messages, tokenize=False, add_generation_prompt=False)
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False).input_ids
    full_ids = tokenizer(full_text, add_special_tokens=False, truncation=True, max_length=max_length).input_ids
    labels = full_ids.copy()
    mask_until = min(len(prompt_ids), len(labels))
    labels[:mask_until] = [-100] * mask_until
    return {"input_ids": full_ids, "attention_mask": [1] * len(full_ids), "labels": labels}

class ShareGPTSFTDataset(Dataset):
    def __init__(self, json_path, tokenizer, max_length, max_samples=None, shuffle=True):
        rows = json.load(open(json_path, encoding="utf-8"))
        rows = [r for r in rows if user_text(r) and assistant_text(r)]
        if shuffle:
            rng = random.Random(SEED)
            rng.shuffle(rows)
        if max_samples:
            rows = rows[:max_samples]
        self.items = [tokenize_chat_example(tokenizer, user_text(r), assistant_text(r), max_length) for r in rows]
        print(f"Loaded {len(self.items)} SFT examples from {json_path}")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        return self.items[idx]

class CausalCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        max_len = max(len(f["input_ids"]) for f in features)
        pad_id = self.tokenizer.pad_token_id
        batch = {"input_ids": [], "attention_mask": [], "labels": []}
        for f in features:
            pad_len = max_len - len(f["input_ids"])
            batch["input_ids"].append(f["input_ids"] + [pad_id] * pad_len)
            batch["attention_mask"].append(f["attention_mask"] + [0] * pad_len)
            batch["labels"].append(f["labels"] + [-100] * pad_len)
        return {k: torch.tensor(v, dtype=torch.long) for k, v in batch.items()}

def load_qwen3_lora_model():
    compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        torch_dtype=compute_dtype,
        quantization_config=bnb_config,
        device_map="auto",
    )
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model)
    lora_config = LoraConfig(
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model, tokenizer

def train_sft(model, tokenizer, json_path, output_dir, epochs, lr, grad_accum=8, max_samples=None):
    dataset = ShareGPTSFTDataset(json_path, tokenizer, CUTOFF_LEN, max_samples=max_samples)
    if len(dataset) == 0:
        print(f"No examples in {json_path}; skipping SFT.")
        return
    args = TrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=epochs,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=grad_accum,
        learning_rate=lr,
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",
        logging_steps=5,
        save_strategy="no",
        report_to="none",
        bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
        fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
        optim="paged_adamw_8bit",
        gradient_checkpointing=True,
        remove_unused_columns=False,
    )
    trainer = Trainer(model=model, args=args, train_dataset=dataset, data_collator=CausalCollator(tokenizer))
    trainer.train()
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def clean_prediction(text):
    text = text.strip()
    for marker in ["<|im_end|>", "</s>", "<|endoftext|>"]:
        text = text.split(marker)[0].strip()
    return text.split("\n")[0].strip()

@torch.no_grad()
def generate_predictions(model, tokenizer, input_json, output_jsonl, max_samples=None, max_new_tokens=16):
    rows = json.load(open(input_json, encoding="utf-8"))
    if max_samples:
        rows = rows[:max_samples]
    model.eval()
    model.config.use_cache = True
    device = next(model.parameters()).device
    output_jsonl = Path(output_jsonl)
    output_jsonl.parent.mkdir(parents=True, exist_ok=True)
    with open(output_jsonl, "w", encoding="utf-8") as f:
        for row in tqdm(rows, desc=f"Generating {Path(input_json).name}"):
            prompt = user_text(row)
            label = assistant_text(row)
            chat = tokenizer.apply_chat_template(
                [{"role": "user", "content": prompt}],
                tokenize=False,
                add_generation_prompt=True,
            )
            inputs = tokenizer(chat, return_tensors="pt", truncation=True, max_length=CUTOFF_LEN).to(device)
            generated = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
            new_tokens = generated[0, inputs["input_ids"].shape[-1]:]
            pred = clean_prediction(tokenizer.decode(new_tokens, skip_special_tokens=True))
            f.write(json.dumps({"predict": pred, "label": label}, ensure_ascii=False) + "\n")
    model.config.use_cache = False
    print(f"Saved predictions to {output_jsonl}")

In [ ]:
#@title 8. Qwen3-4B + LoRA initial SFT
seed_everything(SEED)
model, tokenizer = load_qwen3_lora_model()
train_sft(
    model=model,
    tokenizer=tokenizer,
    json_path=SFT_TRAIN_JSON,
    output_dir=SFT_ADAPTER_DIR,
    epochs=INITIAL_SFT_EPOCHS,
    lr=2e-4,
    grad_accum=8,
)

In [ ]:
#@title 9. Influential node selection + selected dataset
import json
import subprocess
import sys

SELECTED_NODES_JSON = WORK_DIR / f"{RUN_ID}_selected_nodes.json"
select_cmd = [
    sys.executable,
    "select_influential_nodes.py",
    "--dataset", DATASET,
    "--k", str(MAX_SELECTED_NODES),
    "--output_file", str(SELECTED_NODES_JSON),
    "--shots", str(SHOTS),
    "--seed", str(SEED),
    "--method", "auto",
    "--max_subgraph_nodes", "1500",
    "--max_distance", "3",
    "--path_prefix", str(BASE_DIR),
]
subprocess.run(select_cmd, cwd=str(BASE_DIR), check=True)

with open(SELECTED_NODES_JSON) as f:
    selected_ids = set(json.load(f)["selected_node_ids"])
with open(UNLABELED_NODE_IDS_JSON) as f:
    all_unlabeled_ids = json.load(f)["selected_node_ids"]
with open(UNLABELED_JSON, encoding="utf-8") as f:
    unlabeled_rows = json.load(f)

filtered_rows = []
ordered_ids = []
for node_id, row in zip(all_unlabeled_ids, unlabeled_rows):
    if int(node_id) in selected_ids:
        filtered_rows.append(row)
        ordered_ids.append(int(node_id))

SELECTED_DATASET_JSON = DATA_DIR / f"{SFT_PREFIX}_selected.json"
ORDERED_SELECTED_NODES_JSON = WORK_DIR / f"{RUN_ID}_selected_nodes_ordered.json"
with open(SELECTED_DATASET_JSON, "w", encoding="utf-8") as f:
    json.dump(filtered_rows, f, ensure_ascii=False, indent=2)
with open(ORDERED_SELECTED_NODES_JSON, "w", encoding="utf-8") as f:
    json.dump({"selected_node_ids": ordered_ids}, f, indent=2)

print(f"Selected dataset: {len(filtered_rows)} samples")

In [ ]:
#@title 10. LLM predictions on selected nodes
SELECTED_LLM_PRED_JSONL = WORK_DIR / f"{RUN_ID}_selected_llm_preds.jsonl"
generate_predictions(
    model=model,
    tokenizer=tokenizer,
    input_json=SELECTED_DATASET_JSON,
    output_jsonl=SELECTED_LLM_PRED_JSONL,
    max_new_tokens=16,
)

In [ ]:
#@title 11. Committee-of-Judges: ensemble agreement/disagreement → WSFT + DPO
import json
from committee_judge import (
    get_committee_predictions,
    ensemble_find_agreed_and_disagreed,
    filter_disagreed_by_committee_consensus,
    retrain_committee_on_agreed,
    prepare_committee_dpo_dataset,
)
from node_selection import load_llm_predictions_for_selected

# Reload graph data (same split)
from common import create_few_shot_dataset, set_seed
set_seed(SEED)
graph_data = create_few_shot_dataset(
    DATASET, shots=SHOTS, seed=SEED, device=DEVICE, path_prefix=str(BASE_DIR)
)

# Load selected node IDs
with open(ORDERED_SELECTED_NODES_JSON) as f:
    selected_node_ids = json.load(f)["selected_node_ids"]

# Get committee predictions
committee_preds = get_committee_predictions(committee, graph_data)

# Load LLM predictions
llm_predictions = load_llm_predictions_for_selected(
    str(SELECTED_LLM_PRED_JSONL), selected_node_ids, graph_data
)
covered_ids = [int(nid) for nid in selected_node_ids if int(nid) in llm_predictions]
selected_node_ids = covered_ids
print(f"LLM predictions mapped: {len(llm_predictions)}/{len(selected_node_ids)}")

# Find ensemble agreement / disagreement
agreed, disagreed = ensemble_find_agreed_and_disagreed(
    committee_preds, llm_predictions, min_gnn_agreement=MIN_GNN_AGREEMENT,
)
agreed = {nid: agreed[nid] for nid in selected_node_ids if nid in agreed}
disagreed = {nid: disagreed[nid] for nid in selected_node_ids if nid in disagreed}
print(f"Initial committee: {len(agreed)} agreed, {len(disagreed)} disagreed")

# Retrain committee on agreed pseudo labels
if agreed:
    print(f"Retraining committee on {len(agreed)} agreed nodes...")
    retrain_committee_on_agreed(committee, graph_data, agreed, DEVICE, epochs=RETRAIN_EPOCHS)
    committee_preds = get_committee_predictions(committee, graph_data)
    agreed, disagreed = ensemble_find_agreed_and_disagreed(
        committee_preds, llm_predictions, min_gnn_agreement=MIN_GNN_AGREEMENT,
    )
    agreed = {nid: agreed[nid] for nid in selected_node_ids if nid in agreed}
    disagreed = {nid: disagreed[nid] for nid in selected_node_ids if nid in disagreed}
    print(f"After retrain: {len(agreed)} agreed, {len(disagreed)} disagreed")

# Filter disagreed by consensus + variance
final_disagreed = filter_disagreed_by_committee_consensus(
    disagreed,
    confidence_threshold=CONFIDENCE_THRESHOLD,
    variance_threshold=VARIANCE_THRESHOLD,
)
print(f"Disagreed after committee filter: {len(final_disagreed)}")
print(f"  (conf >= {CONFIDENCE_THRESHOLD}, var <= {VARIANCE_THRESHOLD})")

# Generate WSFT + DPO datasets
DPO_JSON = DATA_DIR / f"{RUN_ID}_committee_dpo.json"
WSFT_JSON = DATA_DIR / f"{RUN_ID}_committee_wsft.json"

prepare_committee_dpo_dataset(
    agreed, final_disagreed, graph_data, DATASET, str(DPO_JSON), str(WSFT_JSON),
)

# Summary statistics
wsft_rows = json.load(open(WSFT_JSON, encoding="utf-8"))
dpo_rows = json.load(open(DPO_JSON, encoding="utf-8"))
n_pref = sum(1 for r in dpo_rows if r.get("chosen", {}).get("value") != r.get("rejected", {}).get("value"))
print(f"\n=== Committee-of-Judges Summary ===")
print(f"GNN types: {GNN_TYPES}")
print(f"WSFT examples: {len(wsft_rows)}")
print(f"DPO rows: {len(dpo_rows)} | non-trivial preference pairs: {n_pref}")

In [ ]:
#@title 12. Committee weak-supervised SFT
train_sft(
    model=model,
    tokenizer=tokenizer,
    json_path=WSFT_JSON,
    output_dir=WSFT_ADAPTER_DIR,
    epochs=WSFT_EPOCHS,
    lr=5e-5,
    grad_accum=4,
)

In [ ]:
#@title 13. DPO preference update utilities
class PreferenceDataset(Dataset):
    def __init__(self, json_path, tokenizer, max_length):
        rows = json.load(open(json_path, encoding="utf-8"))
        self.items = []
        for row in rows:
            prompt = user_text(row)
            chosen = row.get("chosen", {}).get("value", "")
            rejected = row.get("rejected", {}).get("value", "")
            if not prompt or not chosen or not rejected or chosen == rejected:
                continue
            self.items.append({
                "chosen": tokenize_chat_example(tokenizer, prompt, chosen, max_length),
                "rejected": tokenize_chat_example(tokenizer, prompt, rejected, max_length),
            })
        print(f"Loaded {len(self.items)} non-trivial preference pairs from {json_path}")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        return self.items[idx]

class PreferenceCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.sft_collator = CausalCollator(tokenizer)

    def __call__(self, features):
        return {
            "chosen": self.sft_collator([f["chosen"] for f in features]),
            "rejected": self.sft_collator([f["rejected"] for f in features]),
        }

def move_batch(batch, device):
    return {k: v.to(device) for k, v in batch.items()}

def sequence_logps(model, batch):
    outputs = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
    logits = outputs.logits[:, :-1, :]
    target = batch["input_ids"][:, 1:]
    loss_mask = batch["labels"][:, 1:] != -100
    log_probs = torch.log_softmax(logits, dim=-1)
    token_logps = torch.gather(log_probs, dim=-1, index=target.unsqueeze(-1)).squeeze(-1)
    return (token_logps * loss_mask).sum(dim=-1)

def run_dpo_update(model, tokenizer, dpo_json, output_dir, epochs=1, beta=0.1, lr=1e-6, grad_accum=2):
    dataset = PreferenceDataset(dpo_json, tokenizer, CUTOFF_LEN)
    if len(dataset) == 0:
        print("No non-trivial preference pairs; skipping DPO stage.")
        return
    loader = DataLoader(dataset, batch_size=1, shuffle=True, collate_fn=PreferenceCollator(tokenizer))
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    device = next(model.parameters()).device
    model.train()
    model.config.use_cache = False
    global_step = 0
    optimizer.zero_grad(set_to_none=True)
    for epoch in range(epochs):
        pbar = tqdm(loader, desc=f"DPO epoch {epoch + 1}/{epochs}")
        for step, batch in enumerate(pbar, start=1):
            chosen = move_batch(batch["chosen"], device)
            rejected = move_batch(batch["rejected"], device)
            with torch.no_grad():
                with model.disable_adapter():
                    ref_chosen = sequence_logps(model, chosen)
                    ref_rejected = sequence_logps(model, rejected)
            pi_chosen = sequence_logps(model, chosen)
            pi_rejected = sequence_logps(model, rejected)
            pi_logratio = pi_chosen - pi_rejected
            ref_logratio = ref_chosen - ref_rejected
            loss = -F.logsigmoid(beta * (pi_logratio - ref_logratio)).mean()
            (loss / grad_accum).backward()
            if step % grad_accum == 0 or step == len(loader):
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1
            pbar.set_postfix(loss=float(loss.detach().cpu()))
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
#@title 14. Optional: DPO preference update
if RUN_DPO_STAGE:
    run_dpo_update(
        model=model,
        tokenizer=tokenizer,
        dpo_json=DPO_JSON,
        output_dir=DPO_ADAPTER_DIR,
        epochs=DPO_EPOCHS,
        beta=0.1,
        lr=1e-6,
        grad_accum=2,
    )
else:
    print("RUN_DPO_STAGE=False; DPO skipped.")

In [ ]:
#@title 15. Final evaluation
import json
import subprocess
import sys

FINAL_TEST_PRED_JSONL = WORK_DIR / "committee_test_predictions.jsonl"
generate_predictions(
    model=model,
    tokenizer=tokenizer,
    input_json=SFT_TEST_JSON,
    output_jsonl=FINAL_TEST_PRED_JSONL,
    max_samples=MAX_TEST_SAMPLES,
    max_new_tokens=16,
)

EVAL_DIR = WORK_DIR / "evaluation"
eval_cmd = [
    sys.executable,
    "evaluate_predictions.py",
    "--dataset", DATASET,
    "--pred_file", str(FINAL_TEST_PRED_JSONL),
    "--output_dir", str(EVAL_DIR),
    "--model_name", "committee_gnn_as_judge",
    "--device", DEVICE,
    "--path_prefix", str(BASE_DIR),
]
subprocess.run(eval_cmd, cwd=str(BASE_DIR), check=True)

metrics_path = EVAL_DIR / "committee_gnn_as_judge" / "metrics.json"
metrics = json.load(open(metrics_path))
print("\n=== Final Results (Committee of GNN Judges) ===")
print(json.dumps(metrics, indent=2))
print("metrics_path =", metrics_path)

In [ ]:
#@title 16. Comparison: committee vs single-judge statistics
print("=" * 60)
print("Committee of GNN Judges Configuration")
print("=" * 60)
print(f"GNN types:             {GNN_TYPES}")
print(f"Min GNN agreement:     {MIN_GNN_AGREEMENT or 'all (unanimous)'}")
print(f"Confidence threshold:  {CONFIDENCE_THRESHOLD}")
print(f"Variance threshold:    {VARIANCE_THRESHOLD}")
print(f"Selected nodes:        {len(selected_node_ids)}")
print(f"Agreed (unanimous):    {len(agreed)}")
print(f"Disagreed (raw):       {len(disagreed)}")
print(f"Disagreed (filtered):  {len(final_disagreed)}")
print(f"WSFT examples:         {len(wsft_rows)}")
print(f"DPO preference pairs:  {n_pref}")
print("=" * 60)
print(f"Accuracy:  {metrics['accuracy']:.4f}")
print(f"Macro-F1:  {metrics['macro_f1']:.4f}")
print("=" * 60)